# Wordspend database audit — 2026-10-22
Run from within the repository using its uv environment. The code below makes only read-only database queries and reads repository manifests. It uses the normal WORDSPEND_DATABASE_URL/default local connection without displaying credentials. The saved output is the October 22 snapshot; rerunning reflects the then-current state.
Assertions verify checksums, contiguous coverage, text spans, and exclusion flags; they do not establish semantic alignment correctness.

In [1]:
import csv
import hashlib
import json
import os
import socket
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import psycopg
from psycopg.rows import dict_row

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/work_catalog.csv').exists())
def rows(relative):
    with (ROOT/relative).open(newline='', encoding='utf-8') as handle:
        return list(csv.DictReader(handle))

queries = {
    'connection': 'SELECT current_database() AS database, current_user AS role;',
    'counts': '''SELECT (SELECT count(*) FROM works) AS works,
        (SELECT count(*) FROM source_text_versions) AS source_versions,
        (SELECT count(*) FROM translations) AS translations,
        (SELECT count(*) FROM aligned_segments) AS aligned_segments,
        (SELECT count(*) FROM analysis_runs) AS analysis_runs,
        (SELECT count(*) FROM segment_residuals) AS residual_rows,
        (SELECT count(*) FROM residual_predictors) AS predictor_rows,
        (SELECT count(*) FROM site_builds) AS site_builds;''',
    'corpus': '''SELECT w.slug,t.translation_source_type,t.is_machine_generated,
        count(s.id) AS segments,min(s.imported_at) AS earliest_import,
        max(s.imported_at) AS latest_import
        FROM aligned_segments s JOIN works w ON w.id=s.work_id
        LEFT JOIN translations t ON t.id=s.translation_id
        GROUP BY w.slug,t.translation_source_type,t.is_machine_generated;''',
    'runs': '''SELECT count(*) AS runs,min(created_at) AS first_run,
        max(created_at) AS last_run,
        count(DISTINCT (intercept,slope,r2)) AS distinct_baselines
        FROM analysis_runs;''',
    'integrity': '''SELECT count(*) AS missing_or_mismatched_links
        FROM aligned_segments s LEFT JOIN source_text_versions v ON v.id=s.source_version_id
        LEFT JOIN translations t ON t.id=s.translation_id
        WHERE v.id IS NULL OR t.id IS NULL OR v.work_id<>s.work_id OR t.work_id<>s.work_id;''',
    'collation': '''SELECT datcollversion AS recorded_version,
        pg_database_collation_actual_version(oid) AS actual_version
        FROM pg_database WHERE datname=current_database();''',
}
audit = {'captured_at':datetime.now(ZoneInfo('Australia/Sydney')).isoformat(), 'host':socket.gethostname(), 'database':{}}
with psycopg.connect(os.environ.get('WORDSPEND_DATABASE_URL','dbname=wordspend'), row_factory=dict_row) as connection:
    connection.execute('SET TRANSACTION READ ONLY')
    for name, query in queries.items():
        audit['database'][name] = connection.execute(query).fetchall()
source = rows('data/prepared/source_editions.csv')
books = rows('data/prepared/translation_books.csv')
candidates = rows('data/prepared/fine_alignment_candidates.csv')
worklists = rows('data/review/fine_alignment_worklists.csv')
verified = 0
for manifest,path_key,hash_key in [
    (source,'prepared_path','prepared_sha256'),
    (books,'book_path','book_sha256'),
    (candidates,'candidate_path','candidate_sha256'),
    (worklists,'review_worklist_path','review_worklist_sha256'),
]:
    for item in manifest:
        assert hashlib.sha256((ROOT/item[path_key]).read_bytes()).hexdigest()==item[hash_key],item[path_key]
        verified+=1
review_rows=[r for w in worklists for r in rows(w['review_worklist_path'])]
assert all(r['paper_facing_eligible']=='no' for r in review_rows)
source_lines={x['source_edition_id']:rows(x['prepared_path']) for x in source}
book_tables={x['source_id']:rows(x['book_path']) for x in books}
for item in candidates:
    proposals=rows(item['candidate_path'])
    assert len(proposals)==int(item['candidate_count'])
    lines=[x for x in source_lines[item['source_edition_id']] if x['book']==item['book']]
    book=next(x for x in book_tables[item['translation_source_id']] if x['book']==item['book'])
    lookup={x['reference']:i for i,x in enumerate(lines)}
    covered=[]
    token_end=0
    for row in proposals:
        span=lines[lookup[row['source_start_ref']]:lookup[row['source_end_ref']]+1]
        covered.extend(span)
        assert row['source_text']=='\n'.join(x['text'] for x in span)
        assert row['target_text']==book['text'][int(row['target_start_char']):int(row['target_end_char'])]
        assert int(row['target_start_token'])==token_end+1
        token_end=int(row['target_end_token'])
        assert row['paper_facing_eligible']=='no'
        assert row['analysis_status']=='excluded_until_human_review'
    assert covered==lines
    assert token_end==int(item['target_token_count'])
audit['files']={
    'translation_registry_rows':len(rows('data/translation_sources.csv')),
    'source_edition_registry_rows':len(rows('data/source_editions.csv')),
    'greek_lines':sum(len(x) for x in source_lines.values()),
    'translation_books':sum(int(x['book_count']) for x in books),
    'translation_tokens':sum(int(x['token_count']) for x in books),
    'structural_book_pairs':len(rows('data/prepared/book_alignments.csv')),
    'candidate_books':len(candidates),
    'candidates':sum(int(x['candidate_count']) for x in candidates),
    'worklist_rows':len(review_rows),
    'entered_review_decisions':sum(bool(x['review_decision'].strip()) for x in review_rows),
    'reviewed_manifest_exists':(ROOT/'data/prepared/reviewed_fine_alignments.csv').exists(),
    'checksums_verified':verified,
    'candidate_coverage_text_and_spans_verified':True,
}
print(json.dumps(audit,indent=2,default=str))


{
  "captured_at": "2026-10-22T11:39:04.916659+11:00",
  "host": "raksasa",
  "database": {
    "connection": [
      {
        "database": "wordspend",
        "role": "gregb"
      }
    ],
    "counts": [
      {
        "works": 44,
        "source_versions": 1,
        "translations": 1,
        "aligned_segments": 3170,
        "analysis_runs": 152,
        "residual_rows": 481840,
        "predictor_rows": 12160,
        "site_builds": 0
      }
    ],
    "corpus": [
      {
        "slug": "pausanias-description-of-greece",
        "translation_source_type": "machine_bootstrap",
        "is_machine_generated": true,
        "segments": 3170,
        "earliest_import": "2026-05-23 14:20:20.694421+10:00",
        "latest_import": "2026-05-23 14:20:20.694421+10:00"
      }
    ],
    "runs": [
      {
        "runs": 152,
        "first_run": "2026-05-23 14:04:12.673388+10:00",
        "last_run": "2026-10-22 04:35:34.121511+11:00",
        "distinct_baselines": 1
      }
    ],


## Separate pipeline smoke check
The unchanged run_daily_pipeline.sh at revision 6ca98a7 was executed against a disposable snapshot of the live schema/corpus, excluding historical output rows. Its rsync destination was a local test directory. The disposable database was removed. This test did not deploy to merah, reimport Pausanias, or change production.
```json
{
  "schema_applied": true,
  "catalog_loaded": 44,
  "analysis_runs_saved": 1,
  "residual_rows_saved": 3170,
  "predictor_rows_saved": 80,
  "static_site_generated": true,
  "rsync_to_local_test_directory": true,
  "production_modified": false
}
```
The read-only live analysis reproduced R² = 0.8623764934745082 across 3,170 bootstrap passages. Local execution logs and the temporary test harness are in the ignored build/database-audit directory. See ../database-audit-2026-10-22.md for deployment inspection and findings.